# 手动标题解析辅助工具 — 实现文档

> 从 B站 IA 视频标题中提取**标准歌名**和**原唱者(P主)**的交互式终端工具。
> 通过建立关键词→歌名+P主的映射数据库，为预处理管道的 `parse_titles()` 步骤提供数据支撑。

## 在项目中的位置

```
爬虫 → music.csv → manual_parser.py → song_keywords.csv → parse_titles()
                      (本工具)            (映射库)          (预处理步骤3)
```

## 设计理念

知名 IA 歌曲反复出现（如"六兆年と一夜物語"有几十个翻唱版本），一次手动标记永久生效。
与其设计越来越复杂的正则，不如建一个关键词映射库，用**子串匹配**实现标题解析。
热门歌看完后，剩下的冷门视频大多能自动命中已有映射。

## 实际架构（700行）

```
manual_parser.py
├── 数据加载层
│   ├── CSV列数探测（表头11列，部分行13列，自动补齐）
│   ├── 宽列名构造 + pd.read_csv(names=..., skiprows=1)
│   └── 按 play_count 降序排列
├── 保存机制（三层防线）
│   ├── _auto_save()         → 每次修改后立即写入磁盘
│   ├── signal(SIGINT/SIGTERM) → Ctrl+C 时紧急保存
│   └── atexit + try/finally  → 进程退出最后兜底
├── 匹配引擎
│   ├── find_matches()       → 子串匹配（大小写不敏感）
│   │     └── 跳过 ≤2 字符纯 ASCII 关键词（防 "IA" 误匹配）
│   ├── _song_index()        → {(歌名, P主): 编号} 查找表
│   └── 匹配缓存             → _row_matched / _row_unmatched 集合
│         └── mapping 修改时自动失效 (_invalidate_cache)
├── 日志系统
│   ├── Tee(stdout)          → 终端 + 文件双写
│   └── data/.parser_log.txt → 含启动时间戳的完整会话日志
└── 交互层
    ├── 主循环（逐行浏览 1~1480）
    └── 管理模式（全部浏览完毕后继续管理映射）
```

## 完整命令表

### 导航命令

| 命令 | 功能 | 说明 |
|------|------|------|
| `s` | 前进一行 | 每 10 行自动保存进度 |
| `b` | 回退一行 | 边界检查（第 1 行不可退） |
| `ss` | 跳到下一个**未匹配**行 | 扫描直到找到无 keyword 命中的行 |
| `bb` | 跳到上一个**未匹配**行 | 向上扫描 |
| `数字` | 跳转到指定行 | 如 `42` → 跳到第 42 行（1-based） |

### 映射命令

| 命令 | 功能 | 说明 |
|------|------|------|
| `n` | 新建歌曲 | 输入关键词(逗号分隔) + 歌名 + P主 |
| `a` | 加到已有歌曲 | 选已有歌曲，追加新关键词 |
| `k` | 别名管理 | 子菜单：增删别名 / 删除整首歌曲 |
| `m` | 修改歌曲 | 改歌名或 P主，批量更新所有关键词 |
| `mg` | 合并歌曲 | 将 B 的所有关键词合并到 A |

### 数据命令

| 命令 | 功能 | 说明 |
|------|------|------|
| `e` | 编辑当前标题 | 修改后写回 music.csv（原子替换） |
| `undo` | 撤销最近映射 | 删除 mapping 最后一行 |

### 系统命令

| 命令 | 功能 | 说明 |
|------|------|------|
| `stat` | 统计 | 映射库规模 + 已匹配/未匹配比例 + 剩余行数 |
| `exp` | 导出 | 子菜单：映射表 / 进度报告 / 两者 |
| `q` | 保存退出 | 确认后保存映射 + 进度 |

### 管理模式下额外可用

| 命令 | 说明 |
|------|------|
| `n` | 新建（手动输入标题用于匹配测试） |
| `a` | 加到已有（无需当前行上下文） |

## 数据加载（处理列数不一致）

music.csv 的特殊问题：表头 11 列，但 950 行有 13 列（分类流水线追加了两个字段）。
`on_bad_lines='skip'` 会将这 950 行全部丢弃。

**解决方案**：先扫描全文件探测最大列数，再构造宽列名读取：

```python
import csv
max_cols = 0
with open(MUSIC_CSV, encoding="utf-8") as f:
    for row in csv.reader(f):
        max_cols = max(max_cols, len(row))

col_names = pd.read_csv(MUSIC_CSV, nrows=0).columns.tolist()
while len(col_names) < max_cols:
    col_names.append(f"_extra_{len(col_names)}")

df = pd.read_csv(MUSIC_CSV, names=col_names, skiprows=1, on_bad_lines="skip")
df = df.sort_values("play_count", ascending=False).reset_index(drop=True)
```

> ⚠️ 教训：`on_bad_lines='skip'` 是静默数据杀手，使用前必须验证数据列数一致性。

## 行展示格式

```
======================================================================
[533/1480]  BV: BV18b411L7ab  |  播放: 7,722
标题: 【IA】春愁【オリジナル曲】
高亮: 【IA】春愁【オリジナル曲】     ← 【】内容黄色高亮
标签: 春愁,IA,Islet
UP主: tayori_official

  ⚡ 已有匹配:
     keyword="春愁" → [#264] 春愁  by Islet    ← 显示歌曲编号
```

- `【】` 内容用 ANSI 黄色高亮，方便在标题中定位虚拟歌姬名字
- 匹配行显示 `[#歌曲编号]`，与 `k`/`m`/`a` 中列表的编号一致
- 标签超过 80 字符截断

## 自动匹配逻辑

```python
def find_matches(title, mapping_df):
    """检查标题是否含已有的 keyword（大小写不敏感）。"""
    matches = []
    for _, row in mapping_df.iterrows():
        kw = str(row["keyword"])
        # 跳过短纯英文关键词（如 "IA"），中文/日文短词保留（如 "春愁"）
        if len(kw) <= 2 and kw.isascii():
            continue
        if kw.upper() in title.upper():
            matches.append(row.to_dict())
    return matches
```

**设计要点**：
- 大小写不敏感（"children record" 匹配 "Children Record"）
- 中文/日文也做子串匹配（"哨戒班" 匹配 "アスノヨゾラ哨戒班"）
- 短纯 ASCII 关键词（如 "IA"、"ok"）跳过——它们会误匹配大量不相关标题
- 中文 2 字符关键词（如 "春愁"）不跳过——CJK 字符误匹配率极低

## 保存机制（三层防线）

标注工作可能持续数小时，单次崩溃就会丢失全部进度。因此设计了多层保存：

```python
# 第1层：每次修改后自动保存
def _auto_save(idx):
    mapping.to_csv(KEYWORD_CSV, index=False, encoding="utf-8-sig")
    with open(PROGRESS_FILE, "w") as f:
        f.write(str(idx))

# 第2层：Ctrl+C / kill 信号
signal.signal(signal.SIGINT, handler)
signal.signal(signal.SIGTERM, handler)

# 第3层：进程退出兜底
atexit.register(_atexit_handler)
# +
try:
    while i < TOTAL: ...
finally:
    if _log_dirty:
        _emergency_save(i)
```

每 10 次 `s` 跳过也触发保存，确保进度文件始终接近最新状态。

## 工作方法建议

1. **先处理播放量高的**：前 200 首热门歌几乎覆盖了所有知名歌曲，后面的 80% 都是冷门
2. **善用 `ss`/`bb`**：按 `ss` 直接跳到下一个需要你判断的行，已匹配的自动跳过
3. **关键词选独特性强的**：'哨戒班' 比 'アスノ' 更独特
4. **用别名统一不同写法**：'Children Record' 和 'チルドレンレコード' 指向同一首歌
5. **定期用 `stat` 查看覆盖率**：80%+ 的已匹配率说明映射库已经很好用
6. **`mg` 合并重复歌曲**：如果发现两首歌其实是同一首（拼写差异），用合并功能

## 实际代码规模（vs 原始预估）

| 组件 | 预估 | 实际 | 说明 |
|------|:--:|:--:|------|
| 数据加载 | ~15行 | ~25行 | 增加了 CSV 列数探测逻辑 |
| 逐行展示 | ~20行 | ~35行 | 增加了【】高亮、歌曲编号显示 |
| 自动匹配 | ~10行 | ~25行 | 增加了 ASCII 短词过滤、匹配缓存 |
| 新建歌曲 | ~15行 | ~40行 | 增加了重复检测、覆盖确认 |
| 加到已有 | ~15行 | ~35行 | 增加了选中确认回显 |
| 主循环 | ~25行 | ~100行 | 增加了 9 个额外命令 |
| 管理模式 | — | ~50行 | 全新模块 |
| 保存机制 | — | ~60行 | signal/atexit/Tee/日志 全新模块 |
| 别名/合并/统计/导出 | — | ~130行 | 全新模块 |
| **总计** | **~100行** | **~700行** | |

> 原始估算严重低估了实际需求。交互式终端工具的健壮性（崩溃恢复、边界处理、数据校验）占了代码量的 40%+。

## 关键 Bug 与修复

| Bug | 现象 | 根因 | 修复 |
|-----|------|------|------|
| 数据丢失 64% | `len(df)=530`，实际 1480 行 | `on_bad_lines='skip'` 静默丢弃列数不一致的行 | 先扫描 CSV 探测最大列数，补齐列名后读取 |
| 保存失败 | 标注数小时的工作全部丢失 | 只有 `q` 退出才保存，Ctrl+C/终端关闭不保存 | signal + atexit + finally 三层防线 |
| "IA" 误匹配 | "IA" 匹配 "Brazil"、"piano" | 短 ASCII 关键词无差别子串匹配 | `find_matches` 跳过 ≤2 字符纯 ASCII 关键词 |
| "春愁" 不匹配 | 2 字中文关键词被跳过 | 上一条修复矫枉过正，跳过了所有短关键词 | 改为只跳过 ASCII 短词，保留 CJK 短词 |
| 标题修改丢失 | `e` 编辑后下次启动恢复原值 | 只改了内存中的 df，没写回磁盘 | `df.to_csv()` 原子替换原文件 |
| stat 性能差 | 每次 stat 扫描全量已浏览行 | O(n×m) 字符串匹配 × 每次调用 | 匹配状态缓存 + mapping 修改时自动失效 |

## 产出文件

| 文件 | 格式 | 内容 |
|------|------|------|
| `data/song_keywords.csv` | CSV | 关键词→标准歌名+原唱者映射库（550+ 条） |
| `data/.parser_progress` | 纯文本 | 当前浏览到的行号（断点续标） |
| `data/.parser_log.txt` | 纯文本 | 完整会话日志（含启动时间戳） |
| `data/parse_progress_report.csv` | CSV | 导出产物：music.csv + 匹配结果列 |

## 运行方式

```bash
cd /path/to/project
python3 manual_parser.py
```

程序自动从上次中断位置继续。进度保存在 `data/.parser_progress`，删除该文件即可从头开始。

## 复杂度评估

| 组件 | 代码量 | 难度 |
|------|:--:|:--:|
| 数据加载 + 排序 | ~15行 | 简单 |
| 逐行展示 | ~20行 | 简单 |
| 自动匹配 | ~10行 | 简单 |
| 新建歌曲 [n] | ~15行 | 简单 |
| 加到已有 [a] | ~15行 | 简单 |
| 主循环 + 进度保存 | ~25行 | 简单 |
| 总计 | **~100行** | 低 |

纯 Python 脚本，无 GUI，终端交互。

额外可选功能（增加复杂度但提升体验）：
- 未保存退出确认（+5行）
- 撤销上一个操作（+15行）
- 查看已有映射统计（+10行）
- 导出标注进度报告（+15行）